In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam

df = pd.read_csv("window_features_clean.csv")
FEATURES = ["hr_mean", "hr_std", "hr_slope", "spo2_mean", "spo2_std", "spo2_slope", "map_mean", "map_std", "map_slope"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["caseid"]))
train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]

# Scaling & Reshaping
scaler = StandardScaler()
X_train_cnn = scaler.fit_transform(train_df[FEATURES]).reshape(-1, len(FEATURES), 1)
X_test_cnn = scaler.transform(test_df[FEATURES]).reshape(-1, len(FEATURES), 1)

def build_cnn_model():
    model = Sequential([
        Input(shape=(len(FEATURES), 1)),
        Conv1D(filters=16, kernel_size=3, activation='relu'),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(16, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(learning_rate=0.005), loss='binary_crossentropy', metrics=['AUC'])
    return model

In [2]:
print("Training 1D CNN: Tachycardia")
y_train_tach = train_df["tachycardia"].values
y_test_tach = test_df["tachycardia"].values

model_tach = build_cnn_model()
model_tach.fit(X_train_cnn, y_train_tach, epochs=15, batch_size=256, verbose=1, validation_split=0.1)

prob_tach = model_tach.predict(X_test_cnn, verbose=0).ravel()
print(f"Tachycardia 1D CNN AUROC: {roc_auc_score(y_test_tach, prob_tach):.4f}")

Training 1D CNN: Tachycardia
Epoch 1/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7485 - loss: 0.5156 - val_AUC: 0.7732 - val_loss: 0.4824
Epoch 2/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7620 - loss: 0.5021 - val_AUC: 0.7758 - val_loss: 0.4792
Epoch 3/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7629 - loss: 0.5005 - val_AUC: 0.7753 - val_loss: 0.4842
Epoch 4/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7640 - loss: 0.5005 - val_AUC: 0.7784 - val_loss: 0.4780
Epoch 5/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7661 - loss: 0.4986 - val_AUC: 0.7750 - val_loss: 0.4802
Epoch 6/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7662 - loss: 0.4988 - val_AUC: 0.7753 - val_loss: 0.4825
Epoch 7/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7671 - loss: 0.4976 - val_AUC: 0.7765 - val_loss: 0.4800
Epoch 8/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7675 - loss: 0.4972 - val_AUC: 0.7741 - val_loss: 0.4823
Epoch 9/15
329/329 ━━━━━━━━

In [3]:
print("Training 1D CNN: Hypotension")
y_train_hypo = train_df["hypotension"].values
y_test_hypo = test_df["hypotension"].values

model_hypo = build_cnn_model()
model_hypo.fit(X_train_cnn, y_train_hypo, epochs=15, batch_size=256, verbose=1, validation_split=0.1)

prob_hypo = model_hypo.predict(X_test_cnn, verbose=0).ravel()
print(f"Hypotension 1D CNN AUROC: {roc_auc_score(y_test_hypo, prob_hypo):.4f}")

Training 1D CNN: Hypotension
Epoch 1/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.6859 - loss: 0.6347 - val_AUC: 0.7168 - val_loss: 0.6109
Epoch 2/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7008 - loss: 0.6231 - val_AUC: 0.7187 - val_loss: 0.6085
Epoch 3/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7052 - loss: 0.6190 - val_AUC: 0.7183 - val_loss: 0.6081
Epoch 4/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7064 - loss: 0.6182 - val_AUC: 0.7186 - val_loss: 0.6079
Epoch 5/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7072 - loss: 0.6178 - val_AUC: 0.7189 - val_loss: 0.6087
Epoch 6/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7085 - loss: 0.6173 - val_AUC: 0.7191 - val_loss: 0.6085
Epoch 7/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7090 - loss: 0.6168 - val_AUC: 0.7212 - val_loss: 0.6059
Epoch 8/15
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.7091 - loss: 0.6163 - val_AUC: 0.7203 - val_loss: 0.6057
Epoch 9/15
329/329 ━━━━━━━━

In [4]:
print("Training 1D CNN: Hypoxia")
y_train_hypox = train_df["hypoxia"].values
y_test_hypox = test_df["hypoxia"].values

neg, pos = np.bincount(y_train_hypox)
class_weight = {0: (1 / neg) * (len(y_train_hypox) / 2.0), 1: (1 / pos) * (len(y_train_hypox) / 2.0)}

model_hypox = build_cnn_model()
model_hypox.fit(X_train_cnn, y_train_hypox, epochs=20, batch_size=256, verbose=1, validation_split=0.1, class_weight=class_weight)

prob_hypox = model_hypox.predict(X_test_cnn, verbose=0).ravel()
print(f"Hypoxia 1D CNN AUROC: {roc_auc_score(y_test_hypox, prob_hypox):.4f}")

Training 1D CNN: Hypoxia
Epoch 1/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.6388 - loss: 0.6495 - val_AUC: 0.6709 - val_loss: 0.6178
Epoch 2/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.6574 - loss: 0.6381 - val_AUC: 0.6694 - val_loss: 0.7034
Epoch 3/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.6621 - loss: 0.6357 - val_AUC: 0.6729 - val_loss: 0.6429
Epoch 4/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.6624 - loss: 0.6351 - val_AUC: 0.6736 - val_loss: 0.6106
Epoch 5/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.6662 - loss: 0.6330 - val_AUC: 0.6743 - val_loss: 0.6160
Epoch 6/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.6689 - loss: 0.6321 - val_AUC: 0.6819 - val_loss: 0.6563
Epoch 7/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.6708 - loss: 0.6299 - val_AUC: 0.6796 - val_loss: 0.6398
Epoch 8/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.6766 - loss: 0.6283 - val_AUC: 0.6876 - val_loss: 0.6462
Epoch 9/20
329/329 ━━━━━━━━━━━━